In [22]:
#Dinamik Fiyatlandırma Motoru

In [23]:
#SABİTLER
MALIYET= 100.0   #Ürünün bize maliyeti fiyat bunun altına inemez
BAZ_FIYAT= 150.0   #Normal piyasa koşullarındaki başlangıç fiyatımız
MAX_FIYAT= 300.0   #Fiyatın çıkabileceği üst tavan

DUSUK_STOK_ESIK= 20      #Bu sayının altındaki stok "az stok" sayılır
YUKSEK_STOK_ESIK= 100     #Bu sayının üstündeki stok "çok stok" sayılır

YUKSEK_TALEP_ESIK= 500    #Bu sayının üstündeki görüntülenme "yüksek talep"
DUSUK_TALEP_ESIK= 100    #Bu sayının altındaki görüntülenme "düşük talep"

In [27]:
#Yardımcı Fonksiyonlar

In [28]:
def talep_faktor_hesapla(goruntulenme: int) -> float:
    #Görüntülenmeye göre fiyat çarpanı döndürür
    if goruntulenme >= YUKSEK_TALEP_ESIK:
        return 1.25   # Yüksek talep → %25 zam
    elif goruntulenme >= DUSUK_TALEP_ESIK:
        return 1.00   # Normal talep → değişiklik yok
    else:
        return 0.90   # Düşük talep → %10 indirim


def stok_faktor_hesapla(stok: int) -> float:
    #Stok miktarına göre fiyat çarpanı döndürür
    if stok <= DUSUK_STOK_ESIK:
        return 1.20   # Az stok → %20 zam
    elif stok <= YUKSEK_STOK_ESIK:
        return 1.00   # Normal stok → değişiklik yok
    else:
        return 0.85   # Çok stok → %15 indirim


def rakip_faktor_hesapla(rakip_fiyati: float) -> float:
    #Rakip fiyatına göre fiyat çarpanı döndürür
    if rakip_fiyati < BAZ_FIYAT * 0.90:
        return 0.95   # Rakip çok ucuz → %5 indirim
    elif rakip_fiyati > BAZ_FIYAT * 1.10:
        return 1.10   # Rakip çok pahalı → %10 zam
    else:
        return 1.00   # Rakip benzer fiyatta → değişiklik yok

In [29]:
#Ana Fonksiyon

In [30]:
def dinamik_fiyat_hesapla(mevcut_stok: int, goruntulenme_24s: int, rakip_fiyati: float)-> float:

    #GİRDİ DOGRULAMA
    if mevcut_stok< 0:
        raise ValueError("Stok miktarı negatif olamaz")
    if goruntulenme_24s< 0:
        raise ValueError("Görüntülenme sayısı negatif olamaz")
    if rakip_fiyati<= 0:
        raise ValueError("Rakip fiyatı sıfır veya negatif olamaz")

    #FAKTOR HESAPLAMA
    talep_faktoru= talep_faktor_hesapla(goruntulenme_24s)
    stok_faktoru= stok_faktor_hesapla(mevcut_stok)
    rakip_faktoru= rakip_faktor_hesapla(rakip_fiyati)

    #FİYAT HESAPLAMA
    yeni_fiyat= BAZ_FIYAT* talep_faktoru* stok_faktoru* rakip_faktoru

    #SİNİR KONTROLU
    yeni_fiyat= max(yeni_fiyat, MALIYET* 1.05)  #en az %5 kâr garantisi
    yeni_fiyat= min(yeni_fiyat, MAX_FIYAT)   #tavana takılma

    return round(yeni_fiyat, 2)

In [31]:
# 4. Senaryo Testleri

In [32]:
print("=" * 50)
print("SENARYO 1: Yüksek Talep, Düşük Stok")
print("Beklenti: Fiyat yükselmeli")
f1= dinamik_fiyat_hesapla(mevcut_stok=10, goruntulenme_24s=800, rakip_fiyati=155)
print(f"Hesaplanan Fiyat: {f1} TL\n")

print("=" * 50)
print("SENARYO 2: Düşük Talep, Yüksek Stok")
print("Beklenti: Fiyat düşmeli")
f2= dinamik_fiyat_hesapla(mevcut_stok=200, goruntulenme_24s=50, rakip_fiyati=145)
print(f"Hesaplanan Fiyat: {f2} TL\n")

print("=" * 50)
print("SENARYO 3: Rakip Çok Ucuz, Normal Koşullar")
print("Beklenti: Fiyat rakibe yaklaşmalı ama zarara girmemeli")
f3= dinamik_fiyat_hesapla(mevcut_stok=60, goruntulenme_24s=200, rakip_fiyati=100)
print(f"Hesaplanan Fiyat: {f3} TL\n")

print("=" * 50)
print("SENARYO 4 (BONUS): Hata Yönetimi Testi")
try:
    dinamik_fiyat_hesapla(mevcut_stok=-5, goruntulenme_24s=300, rakip_fiyati=150)
except ValueError as e:
    print(f"Hata yakalandı (beklenen): {e}")

SENARYO 1: Yüksek Talep, Düşük Stok
Beklenti: Fiyat yükselmeli
Hesaplanan Fiyat: 225.0 TL

SENARYO 2: Düşük Talep, Yüksek Stok
Beklenti: Fiyat düşmeli
Hesaplanan Fiyat: 114.75 TL

SENARYO 3: Rakip Çok Ucuz, Normal Koşullar
Beklenti: Fiyat rakibe yaklaşmalı ama zarara girmemeli
Hesaplanan Fiyat: 142.5 TL

SENARYO 4 (BONUS): Hata Yönetimi Testi
Hata yakalandı (beklenen): Stok miktarı negatif olamaz
